In [1]:
import os, glob
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from tqdm import tqdm


In [2]:
obs_data_path = r"D:\!Work\Moscow\!data\meteo_data\msu\data_msu_fixed.csv"

msu_time_belt = 3

df_msu = pd.read_csv (obs_data_path, sep = ';', parse_dates=['DATA'], dayfirst=True).rename(columns={'T33': 'rh', 'TB':'ta', 'PPP':'ps'})[['DATA', 'SR', 'ta', 'rh', 'ps']]

df_msu['DATA'] = df_msu['DATA'] + pd.Series ([pd.Timedelta(hours=x) for x in df_msu['SR']], index = df_msu.index) - pd.Timedelta (hours=msu_time_belt)

df_msu = df_msu.set_index ('DATA')
df_msu['rho'] = (df_msu['ps']*100) / (286.7 * (df_msu['ta'] + 273.15))
df_msu.head()

C:\Users\mvar9\AppData\Local\Temp\ipykernel_14628\3547981387.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_msu = pd.read_csv (obs_data_path, sep = ';', parse_dates=['DATA'], dayfirst=True).rename(columns={'T33': 'rh', 'TB':'ta', 'PPP':'ps'})[['DATA', 'SR', 'ta', 'rh', 'ps']]


,SR,ta,rh,ps,rho
DATA,,,,,
1976-12-31 21:00:00,0,-14.7,66.0,1015.4,1.370354
1977-01-01 00:00:00,3,-15.2,69.0,1016.3,1.374228
1977-01-01 03:00:00,6,-15.1,78.0,1016.8,1.374371
1977-01-01 06:00:00,9,-15.7,69.0,1017.6,1.378658
1977-01-01 09:00:00,12,-14.4,59.0,1018.7,1.373214


In [3]:
twr_data_dir = r"D:\!Work\IFA\!data\Urban_fluxes_NE\raw\Moscow\MSU mast\FLUXES_MSU"

sites = ['A1', 'A2', 'A3']

levels = {'A1': 2.2, 'A2': 11.1, 'A3': 18.8}

df4sites = {}
for site in sites:
    df4site = pd.DataFrame()
    files = glob.glob (twr_data_dir + '/' + site + '/*/*turb_stats*.csv')
    for file in tqdm (files):
        cur_df = pd.read_csv (file, sep = ',', parse_dates=[0], index_col=[0])
        df4site = pd.concat((df4site, cur_df))

    time_arr = pd.date_range(pd.Timestamp(df4site.index.date[0]), pd.Timestamp(df4site.index.date[-1]), freq = '30min') 
    df4site = df4site.resample('30min').last() 
    df4site = df4site.reindex(time_arr)

    df_msu_sel = df_msu[df4site.index[0]:df4site.index[-1]].resample('30min').last().interpolate(method='linear')
    df4site = df4site.merge(df_msu_sel, how = 'left', left_index = True, right_index = True)

    df4site['H'] = df4site['wt'] * df4site['rho'] * 1003
    df4site['tau'] = df4site['u_star']**2 * df4site['rho'] 
    df4site['wind_vel'] = np.abs(df4site['u_rot'])
    df4site['sigma_U'] = np.sqrt(df4site['uu'])
    df4site['sigma_V'] = np.sqrt(df4site['vv'])
    df4site['sigma_W'] = np.sqrt(df4site['ww'])

    df4sites[str(int (np.round (levels[site]))) + 'm'] = df4site

    #display(files)

  0%|          | 0/36 [00:00<?, ?it/s]

100%|██████████| 30/30 [00:00<00:00, 46.62it/s]


In [4]:
# %matplotlib qt
# df4sites['A1']['tau'].resample('D').mean().plot()
# df4sites['A2']['tau'].resample('D').mean().plot()
# df4sites['A3']['tau'].resample('D').mean().plot()

In [5]:
# df4sites['A1']['2022':'2023']['t'].plot()
# df4sites['A2']['2022':'2023']['t'].plot()
# df4sites['A3']['2022':'2023']['t'].plot()
# df_msu['2022':'2023']['ta'].plot()

In [6]:
(df4site['sigma_W'] / df4site['u_star']).median()

1.2985616543893164

In [8]:
export_dir = "D:/!Work/IFA/!data/Urban_fluxes_NE/processed/Moscow"
os.makedirs(export_dir, exist_ok=True)

for level in df4sites.keys():
    df4site = df4sites[level]
    df2export = df4site[['H', 'tau', 'u_star', 'ta', 'rh', 'ps', 'rho', 'u', 'v', 'w', 'sigma_U', 'sigma_V', 'sigma_W', 'wind_vel']]
    df2export = df2export.rename(columns = {'H': 'Qh',
                                            'tau': 'Qtau',
                                            'u_star':'Ustar', 
                                            'ta': 'T_air',
                                            'rh': 'RH_air',
                                            'ps': 'P_air',
                                            'rho': 'Rho_air',
                                            'v':'Wind_V',
                                            'u':'Wind_U',
                                            'w':'Wind_W',
                                            'wind_vel':'Wind_vel',
                                            'sigma_U':'Wind_U_std',
                                            'sigma_V':'Wind_V_std', 
                                            'sigma_W':'Wind_W_std'})
    df2export.index = df2export.index
    df2export.to_csv(export_dir + '/' + 'MSU_' +level +  '.csv')
